<a href="https://colab.research.google.com/github/savreet-088/Cloud/blob/main/cloud_lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1. Word Count - Count the frequency of each word
Input:
• hadoop is fast
• hadoop is scalable

In [ ]:
data = ["hadoop is fast", "hadoop is scalable"]

Manual

In [ ]:
def mapper(data):
    mapped = []
    for line in data:
        for word in line.split():
            mapped.append((word, 1))
    return mapped

In [ ]:
from collections import defaultdict

def shuffle(mapped):
    grouped = defaultdict(list)
    for key, value in mapped:
        grouped[key].append(value)
    return grouped

In [ ]:
def reducer(grouped):
    return {k: sum(v) for k, v in grouped.items()}

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

{'hadoop': 2, 'is': 2, 'fast': 1, 'scalable': 1}


mrjob

In [ ]:
!pip install mrjob

In [ ]:
from mrjob.job import MRJob

In [ ]:
%%writefile wordcount.py

from mrjob.job import MRJob

class WordCount(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield word, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    WordCount.run()

Writing wordcount.py


In [ ]:
%%writefile input.txt
hadoop is fast
hadoop is scalable

Writing input.txt


In [ ]:
!python wordcount.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/wordcount.root.20260420.061510.339288
Running step 1 of 1...
job output is in /tmp/wordcount.root.20260420.061510.339288/output
Streaming final output from /tmp/wordcount.root.20260420.061510.339288/output...
"is"	2
"scalable"	1
"fast"	1
"hadoop"	2
Removing temp directory /tmp/wordcount.root.20260420.061510.339288...


Q2. Character Count - Count the frequency of each character (ignore spaces).
Input: big data

manual

In [ ]:
data = ["big data"]

def mapper(data):
    mapped = []
    for line in data:
        for ch in line.replace(" ", ""):
            mapped.append((ch, 1))
    return mapped

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

{'b': 1, 'i': 1, 'g': 1, 'd': 1, 'a': 2, 't': 1}


mrjob

In [ ]:
%%writefile q2_charcount.py
from mrjob.job import MRJob

class CharCount(MRJob):
    def mapper(self, _, line):
        # Clean line: remove spaces & newline, make lowercase
        line = line.strip().replace(" ", "").lower()

        for ch in line:
            yield ch, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    CharCount.run()

Writing q2_charcount.py


In [ ]:
%%writefile input2.txt
big data

Writing input2.txt


In [ ]:
!python q2_charcount.py input2.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q2_charcount.root.20260420.061510.858627
Running step 1 of 1...
job output is in /tmp/q2_charcount.root.20260420.061510.858627/output
Streaming final output from /tmp/q2_charcount.root.20260420.061510.858627/output...
"b"	1
"d"	1
"g"	1
"i"	1
"t"	1
"a"	2
Removing temp directory /tmp/q2_charcount.root.20260420.061510.858627...


Q3.Average Word Length (Per Word) - Compute the average length of each word.
Input: data science data big data

manual

In [ ]:
data = ["data science data big data"]

def mapper(data):
    mapped = []
    for line in data:
        for word in line.split():
            mapped.append((word, len(word)))
    return mapped

def reducer(grouped):
    return {k: sum(v)/len(v) for k, v in grouped.items()}

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

{'data': 4.0, 'science': 7.0, 'big': 3.0}


mrjob

In [ ]:
%%writefile q3_avg_word_len.py
from mrjob.job import MRJob

class AvgWordLength(MRJob):

    def mapper(self, _, line):
        for word in line.strip().split():
            yield word.lower(), (len(word), 1)

    def reducer(self, key, values):
        total_len = 0
        total_count = 0

        for length, count in values:
            total_len += length
            total_count += count

        yield key, total_len / total_count

if __name__ == "__main__":
    AvgWordLength.run()

Writing q3_avg_word_len.py


In [ ]:
%%writefile input3.txt
data science data big data

Writing input3.txt


In [ ]:
!python q3_avg_word_len.py input3.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q3_avg_word_len.root.20260420.061511.319667
Running step 1 of 1...
job output is in /tmp/q3_avg_word_len.root.20260420.061511.319667/output
Streaming final output from /tmp/q3_avg_word_len.root.20260420.061511.319667/output...
"science"	7.0
"big"	3.0
"data"	4.0
Removing temp directory /tmp/q3_avg_word_len.root.20260420.061511.319667...


Q4.Global Average Word Length - Compute the average length of all words.
Input: hadoop mapreduce spark

manual

In [ ]:
data = ["hadoop mapreduce spark"]

def mapper(data):
    mapped = []
    for line in data:
        for word in line.split():
            mapped.append(("all", len(word)))
    return mapped

def reducer(grouped):
    total = sum(grouped["all"])
    count = len(grouped["all"])
    return total / count

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

6.666666666666667


mrjob

In [ ]:
%%writefile q4_global_avg.py
from mrjob.job import MRJob

class GlobalAvgWordLength(MRJob):

    def mapper(self, _, line):
        for word in line.strip().split():
            # emit (key, (length, count))
            yield "global", (len(word), 1)

    def reducer(self, key, values):
        total_length = 0
        total_count = 0

        for length, count in values:
            total_length += length
            total_count += count

        yield key, total_length / total_count

if __name__ == "__main__":
    GlobalAvgWordLength.run()

Writing q4_global_avg.py


In [ ]:
%%writefile input4.txt
hadoop mapreduce spark

Writing input4.txt


In [ ]:
!python q4_global_avg.py input4.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q4_global_avg.root.20260420.061511.763357
Running step 1 of 1...
job output is in /tmp/q4_global_avg.root.20260420.061511.763357/output
Streaming final output from /tmp/q4_global_avg.root.20260420.061511.763357/output...
"global"	6.666666666666667
Removing temp directory /tmp/q4_global_avg.root.20260420.061511.763357...


Q5. Perform Q1-Q4 on the file
Also find Top 5 most frequent words

manual

In [ ]:
from collections import Counter

with open("/content/shakespeare.txt") as f:
    data = f.readlines()

mapped = []
for line in data:
    for word in line.split():
        mapped.append((word.lower(), 1))

def reducer(grouped):
    result = {}
    for key, values in grouped.items():
        result[key] = sum(values)
    return result


grouped = shuffle(mapped)
word_count = reducer(grouped)

top = Counter(word_count).most_common(5)

print(top)



[('the', 27729), ('and', 26099), ('i', 19540), ('to', 18762), ('of', 18126)]


mrjob

In [1]:
!pip install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 23.0 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving shakespeare.txt to shakespeare.txt


In [3]:
%%writefile mrjob_analysis.py
from mrjob.job import MRJob
from mrjob.step import MRStep
import re

class MRFullAnalysis(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper_all,
                   reducer=self.reducer_step1),
            MRStep(reducer=self.reducer_step2)
        ]

    def mapper_all(self, _, line):
        words = re.findall(r'\w+', line.lower())

        for word in words:
            yield ("word_freq", word), 1
            yield ("word_len", word), len(word)

        for char in line.replace(" ", "").lower():
            yield ("char_freq", char), 1

        for word in words:
            yield ("global_sum", "sum"), len(word)
            yield ("global_sum", "count"), 1

    def reducer_step1(self, key, values):
        yield key, sum(values)

    def reducer_step2(self, key, values):
        yield key, sum(values)

if __name__ == '__main__':
    MRFullAnalysis.run()

Writing mrjob_analysis.py


In [4]:
!python mrjob_analysis.py your_file.txt > output.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Traceback (most recent call last):
  File "/content/mrjob_analysis.py", line 35, in <module>
    MRFullAnalysis.run()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/job.py", line 616, in run
    cls().execute()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/job.py", line 687, in execute
    self.run_job()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/job.py", line 636, in run_job
    runner.run()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/runner.py", line 500, in run
    self._check_input_paths()
  File "/usr/local/lib/python3.12/dist-packages/mrjob/runner.py", line 1133, in _check_input_paths
    self._check_input_path(path)
  File "/usr/local/lib/python3.12/dist-packages/mrjob/runner.py", line 1146, in _check_input_path
    raise IOError(
OSError: Input path your_file.txt does not exist!


In [5]:
with open("output.txt") as f:
    print(f.read())

In [6]:
import ast
from collections import Counter

word_counts = {}

with open("output.txt") as f:
    for line in f:
        key, value = line.strip().split('\t')
        key = ast.literal_eval(key)
        value = int(value)

        if key[0] == "word_freq":
            word_counts[key[1]] = value

top5 = Counter(word_counts).most_common(5)
print("Top 5 words:", top5)

Top 5 words: []


Q6. Compute average marks for each student.
Input:
A 80
B 70
A 90
B 60
A 100

manual

In [ ]:
data = ["A 80", "B 70", "A 90", "B 60", "A 100"]

def mapper(data):
    mapped = []
    for line in data:
        name, marks = line.split()
        mapped.append((name, int(marks)))
    return mapped

def reducer(grouped):
    return {k: sum(v)/len(v) for k, v in grouped.items()}

mapped = mapper(data)
grouped = shuffle(mapped)
result = reducer(grouped)

print(result)

{'A': 90.0, 'B': 65.0}


mrjob

In [7]:
%%writefile q6_avg_marks.py
from mrjob.job import MRJob

class MRAverageMarks(MRJob):

    def mapper(self, _, line):
        student, marks = line.split()
        yield student, int(marks)

    def reducer(self, student, marks):
        marks = list(marks)
        avg = sum(marks) / len(marks)
        yield student, avg

if __name__ == '__main__':
    MRAverageMarks.run()

Writing q6_avg_marks.py


In [10]:
%%writefile input6.txt
A 80
B 70
A 90
B 60
A 100

Writing input6.txt


In [11]:
!python q6_avg_marks.py input6.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q6_avg_marks.root.20260428.045940.459761
Running step 1 of 1...
job output is in /tmp/q6_avg_marks.root.20260428.045940.459761/output
Streaming final output from /tmp/q6_avg_marks.root.20260428.045940.459761/output...
"B"	65.0
"A"	90.0
Removing temp directory /tmp/q6_avg_marks.root.20260428.045940.459761...


Q7.Compute average salary per department and Highest Paid Department (Based on Average
Salary)
Input:
HR 50000
IT 70000
HR 60000
IT 80000

manual

In [ ]:
data = ["HR 50000", "IT 70000", "HR 60000", "IT 80000"]

mapped = [(line.split()[0], int(line.split()[1])) for line in data]

grouped = shuffle(mapped)

avg_salary = {k: sum(v)/len(v) for k, v in grouped.items()}

highest = max(avg_salary, key=avg_salary.get)

print("Average:", avg_salary)
print("Highest Paid Dept:", highest)

Average: {'HR': 55000.0, 'IT': 75000.0}
Highest Paid Dept: IT


mrjob

In [12]:
%%writefile q7_salary.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class MRSalaryAnalysis(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer_avg),
            MRStep(reducer=self.reducer_max)
        ]

    # Mapper
    def mapper(self, _, line):
        dept, salary = line.split()
        yield dept, int(salary)

    # Step 1: Compute average salary per department
    def reducer_avg(self, dept, salaries):
        salaries = list(salaries)
        avg = sum(salaries) / len(salaries)
        yield None, (dept, avg)

    # Step 2: Find highest paid department
    def reducer_max(self, _, dept_avgs):
        max_dept = None
        max_avg = 0

        for dept, avg in dept_avgs:
            yield dept, avg   # output each dept avg
            if avg > max_avg:
                max_avg = avg
                max_dept = dept

        yield "Highest Paid Department", (max_dept, max_avg)

if __name__ == '__main__':
    MRSalaryAnalysis.run()

Writing q7_salary.py


In [13]:
%%writefile input7.txt
HR 50000
IT 70000
HR 60000
IT 80000

Writing input7.txt


In [14]:
!python q7_salary.py input7.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q7_salary.root.20260428.050137.433007
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/q7_salary.root.20260428.050137.433007/output
Streaming final output from /tmp/q7_salary.root.20260428.050137.433007/output...
"IT"	75000.0
"HR"	55000.0
"Highest Paid Department"	["IT", 75000.0]
Removing temp directory /tmp/q7_salary.root.20260428.050137.433007...


Q8.Computer average temperature per country
New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35

manual

In [ ]:
data = [
    "New York,38", "London,29", "Tokyo,35",
    "New York,32", "Delhi,45", "Ambala,35"
]

mapped = []
for line in data:
    city, temp = line.split(",")
    mapped.append((city, int(temp)))

grouped = shuffle(mapped)

result = {k: sum(v)/len(v) for k, v in grouped.items()}

print(result)

{'New York': 35.0, 'London': 29.0, 'Tokyo': 35.0, 'Delhi': 45.0, 'Ambala': 35.0}


mrjob

In [15]:
%%writefile q8_temp.py
from mrjob.job import MRJob

class MRAvgTemp(MRJob):

    # Mapper
    def mapper(self, _, line):
        city, temp = line.split(',')
        yield city, int(temp)

    # Reducer
    def reducer(self, city, temps):
        temps = list(temps)
        avg = sum(temps) / len(temps)
        yield city, avg

if __name__ == '__main__':
    MRAvgTemp.run()

Writing q8_temp.py


In [16]:
%%writefile input8.txt
New York,38
London,29
Tokyo,35
New York,32
Delhi,45
Ambala,35

Writing input8.txt


In [17]:
!python q8_temp.py input8.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q8_temp.root.20260428.050257.272475
Running step 1 of 1...
job output is in /tmp/q8_temp.root.20260428.050257.272475/output
Streaming final output from /tmp/q8_temp.root.20260428.050257.272475/output...
"London"	29.0
"New York"	35.0
"Tokyo"	35.0
"Ambala"	35.0
"Delhi"	45.0
Removing temp directory /tmp/q8_temp.root.20260428.050257.272475...


Q9. Redo on dataset

In [ ]:
import pandas as pd

df = pd.read_csv("GlobalLandTemperaturesByCity.csv")

# Clean
df = df.dropna()
df = df[['City', 'AverageTemperature']]

# MapReduce style
mapped = list(zip(df['City'], df['AverageTemperature']))
grouped = shuffle(mapped)

result = {k: sum(v)/len(v) for k, v in grouped.items()}

FileNotFoundError: [Errno 2] No such file or directory: 'GlobalLandTemperaturesByCity.csv'